# 01 - Data Cleaning

... 

## Examine columns and do team check

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/teams_game_level.csv')

print(df.shape)
print(df['team'].unique())

for col in df.columns:
    print(col)

(232220, 111)
<StringArray>
['NYR', 'SJS', 'PIT', 'EDM', 'WPG', 'PHI', 'DAL', 'N.J', 'T.B', 'MIN', 'COL',
 'ANA', 'VGK', 'TOR', 'NYI', 'MTL', 'SEA', 'STL', 'FLA', 'BUF', 'UTA', 'LAK',
 'ARI', 'OTT', 'WSH', 'NJD', 'TBL', 'CBJ', 'CHI', 'NSH', 'CGY', 'BOS', 'DET',
 'VAN', 'CAR', 'S.J', 'ATL', 'L.A']
Length: 38, dtype: str
team
season
name
gameId
playerTeam
opposingTeam
home_or_away
gameDate
position
situation
xGoalsPercentage
corsiPercentage
fenwickPercentage
iceTime
xOnGoalFor
xGoalsFor
xReboundsFor
xFreezeFor
xPlayStoppedFor
xPlayContinuedInZoneFor
xPlayContinuedOutsideZoneFor
flurryAdjustedxGoalsFor
scoreVenueAdjustedxGoalsFor
flurryScoreVenueAdjustedxGoalsFor
shotsOnGoalFor
missedShotsFor
blockedShotAttemptsFor
shotAttemptsFor
goalsFor
reboundsFor
reboundGoalsFor
freezeFor
playStoppedFor
playContinuedInZoneFor
playContinuedOutsideZoneFor
savedShotsOnGoalFor
savedUnblockedShotAttemptsFor
penaltiesFor
penalityMinutesFor
faceOffsWonFor
hitsFor
takeawaysFor
giveawaysFor
lowDangerShotsFor


In [2]:
# Check which seasons each code variant appears in
for pair in [('N.J', 'NJD'), ('T.B', 'TBL'), ('L.A', 'LAK'), ('S.J', 'SJS')]:
    old, new = pair
    old_seasons = sorted(df[df['team'] == old]['season'].unique())
    new_seasons = sorted(df[df['team'] == new]['season'].unique())
    print(f"{old}: {old_seasons}")
    print(f"{new}: {new_seasons}")
    print()

N.J: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
NJD: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

T.B: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
TBL: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

L.A: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
LAK: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

S.J: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(20

In [3]:
# Standardize team codes - fix formatting inconsistencies (pre 2021) and relocations
team_code_map = {
    'N.J': 'NJD',
    'T.B': 'TBL',
    'L.A': 'LAK',
    'S.J': 'SJS',
    'ATL': 'WPG',  # Atlanta Thrashers -> Winnipeg Jets (2011)
    'ARI': 'UTA',  # Arizona Coyotes -> Utah (2024)
}

df['team'] = df['team'].replace(team_code_map)
df['opposingTeam'] = df['opposingTeam'].replace(team_code_map)

print(df['team'].nunique(), 'unique teams after cleanup')
print(sorted(df['team'].unique()))

32 unique teams after cleanup
['ANA', 'BOS', 'BUF', 'CAR', 'CBJ', 'CGY', 'CHI', 'COL', 'DAL', 'DET', 'EDM', 'FLA', 'LAK', 'MIN', 'MTL', 'NJD', 'NSH', 'NYI', 'NYR', 'OTT', 'PHI', 'PIT', 'SEA', 'SJS', 'STL', 'TBL', 'TOR', 'UTA', 'VAN', 'VGK', 'WPG', 'WSH']


## Generate win/loss labels

`teams_game_level.csv` has no outcome column, and its own `home_or_away` field is
unreliable for ~115 games (both rows sometimes show 'AWAY'). Instead, we pull
`homeTeamWon` and `homeTeamCode` from the shots file - the same source `homeTeamWon`
was computed from - and derive win/loss from that directly. This also correctly
resolves shootout-decided games, which show equal `goalsFor`/`goalsAgainst` in the
team-game file and can't be resolved from goals alone.

In [4]:
# Filter to full-game totals (situation == 'all' sums every strength situation,
# giving one row per team per game with the true final score)
all_situation = df[df['situation'] == 'all'].copy()
print(all_situation.shape)

(46444, 111)


In [5]:
# Load shot data - used only to pull each game's outcome (homeTeamWon, homeTeamCode)
shots = pd.read_csv(
    '../data/raw/shots_2007-2024.csv',
    usecols=['game_id', 'homeTeamCode', 'homeTeamWon', 'season']
)
print(shots.shape)

(1960088, 4)


In [6]:
# shots' game_id lacks the season prefix and leading zero on game type
# (e.g. 20001 -> should be 2008020001). Reconstruct to match teams_game_level format.
shots['gameId'] = shots['season'] * 1_000_000 + shots['game_id']

game_outcomes = shots.drop_duplicates('gameId')[['gameId', 'homeTeamWon', 'homeTeamCode']].copy()

# homeTeamCode also needs the same team-code standardization applied earlier
game_outcomes['homeTeamCode'] = game_outcomes['homeTeamCode'].replace(team_code_map)

print(game_outcomes.shape)

(23142, 3)


In [7]:
all_situation = all_situation.merge(game_outcomes, on='gameId', how='left')
print(all_situation['homeTeamWon'].isna().sum(), 'unmatched rows out of', len(all_situation))

# Check which seasons the unmatched rows belong to
unmatched = all_situation[all_situation['homeTeamWon'].isna()]
print(unmatched['season'].value_counts().sort_index())

2790 unmatched rows out of 46444
season
2021       2
2025    2788
Name: count, dtype: int64


In [8]:
all_situation = all_situation[all_situation['homeTeamWon'].notna()].copy()
print(all_situation.shape)

(43654, 113)


In [10]:
# win = 1 if (this team is home AND home won) OR (this team is away AND home lost)
all_situation['win'] = (
    (all_situation['team'] == all_situation['homeTeamCode']) == (all_situation['homeTeamWon'] == 1)
).astype(int)

print(all_situation['win'].value_counts())

win
1    21827
0    21827
Name: count, dtype: int64
